In [9]:
import pandas as pd

# 1. Load the labeled dataset
df = pd.read_excel('labeled_data.xlsx')

# 2. Frequency count of each Orientation
orientation_counts = df['Group_Orientation'].value_counts()
print("=== Frequency of Orientations ===")
print(orientation_counts)

# 3. Percentage of each Orientation
orientation_percent = df['Group_Orientation'].value_counts(normalize=True) * 100
print("\n=== Percentage of Orientations ===")
print(orientation_percent.round(1))

=== Frequency of Orientations ===
OUT    327
NE     153
IN     139
Name: Group_Orientation, dtype: int64

=== Percentage of Orientations ===
OUT    52.8
NE     24.7
IN     22.5
Name: Group_Orientation, dtype: float64


In [10]:
contingency = pd.crosstab(df['Identity_Dimension'], df['Group_Orientation'])
print(contingency)

Group_Orientation   IN  NE  OUT
Identity_Dimension             
C                    1   0    3
CU                   0   3    3
E                    0   0    3
ECO                  0   4    1
EDU                  0   1    1
F                    0   1    0
G                    7   0    5
GENERATIONAL         1   3    1
I                    3   7    0
M                    2   9    8
N                   17   7   44
O                    4   2   13
P                   56  22  170
R                   33   6   26
S                    0   2    4
SPORT                3   1    4
S_C                  0   2    8
U                   12  83   33


In [11]:
import pandas as pd

# Load the main dataset
df = pd.read_excel('labeled_data.xlsx')


# FIRST: Create the full contingency table with Total
contingency_full = pd.crosstab(df['Identity_Dimension'], df['Group_Orientation'])
contingency_full['Total'] = contingency_full.sum(axis=1)


# THEN: Keep only dimensions with total count >= 10
keep_dims = contingency_full[contingency_full['Total'] >= 10].index.tolist()
print("\n=== Retained dimensions for analysis ===")
print(keep_dims)

# Filter the main dataframe (only dimensions with Total >= 10)
df_filtered = df[df['Identity_Dimension'].isin(keep_dims)]

# New contingency table (only for analysis)
contingency_filtered = pd.crosstab(df_filtered['Identity_Dimension'], df_filtered['Group_Orientation'])
print("\n=== Filtered Contingency Table (only dimensions with Total >= 10) ===")
print(contingency_filtered)

# Save the new table for the article
contingency_filtered.to_excel('contingency_filtered.xlsx')
print("\n✅ Filtered contingency table saved as 'contingency_filtered.xlsx'")


=== Retained dimensions for analysis ===
['G', 'I', 'M', 'N', 'O', 'P', 'R', 'S_C', 'U']

=== Filtered Contingency Table (only dimensions with Total >= 10) ===
Group_Orientation   IN  NE  OUT
Identity_Dimension             
G                    7   0    5
I                    3   7    0
M                    2   9    8
N                   17   7   44
O                    4   2   13
P                   56  22  170
R                   33   6   26
S_C                  0   2    8
U                   12  83   33

✅ Filtered contingency table saved as 'contingency_filtered.xlsx'


In [12]:
from scipy.stats import chi2_contingency, fisher_exact
import itertools

# جدول توافقی فیلتر شده شما
contingency_filtered = pd.crosstab(df_filtered['Identity_Dimension'], df_filtered['Group_Orientation'])

# ====================
# 1. Chi-square test (کل جدول)
# ====================
chi2, p_chi2, dof, expected = chi2_contingency(contingency_filtered)
print("=== Chi-square Test (Full Table) ===")
print(f"p-value = {p_chi2:.5f}")
if p_chi2 < 0.05:
    print("✅ There is a significant relationship between identity dimension and group orientation.\n")
else:
    print("❌ No significant relationship found.\n")

# ====================
# 2. Fisher's Exact Test (pairwise comparisons)
# ====================
print("=== Pairwise Fisher's Exact Tests (OUT vs NE) ===\n")

# گرفتن لیست ابعاد
dims = contingency_filtered.index.tolist()

# مقایسه هر جفت بعد
for dim1, dim2 in itertools.combinations(dims, 2):
    out1 = contingency_filtered.loc[dim1, 'OUT']
    ne1 = contingency_filtered.loc[dim1, 'NE']
    out2 = contingency_filtered.loc[dim2, 'OUT']
    ne2 = contingency_filtered.loc[dim2, 'NE']
    
    # فقط مقایسه‌هایی که مجموع OUT+NE حداقل ۵ دارند
    if (out1 + ne1) >= 5 and (out2 + ne2) >= 5:
        table = [[out1, ne1], [out2, ne2]]
        odds, p = fisher_exact(table)
        if p < 0.05:
            print(f"{dim1} vs {dim2}: p = {p:.5f} → Significant ✅")
        else:
            print(f"{dim1} vs {dim2}: p = {p:.5f} → Not significant ❌")

=== Chi-square Test (Full Table) ===
p-value = 0.00000
✅ There is a significant relationship between identity dimension and group orientation.

=== Pairwise Fisher's Exact Tests (OUT vs NE) ===

G vs I: p = 0.00126 → Significant ✅
G vs M: p = 0.05366 → Not significant ❌
G vs N: p = 1.00000 → Not significant ❌
G vs O: p = 1.00000 → Not significant ❌
G vs P: p = 1.00000 → Not significant ❌
G vs R: p = 0.56689 → Not significant ❌
G vs S_C: p = 0.52381 → Not significant ❌
G vs U: p = 0.00252 → Significant ✅
I vs M: p = 0.05379 → Not significant ❌
I vs N: p = 0.00001 → Significant ✅
I vs O: p = 0.00021 → Significant ✅
I vs P: p = 0.00000 → Significant ✅
I vs R: p = 0.00011 → Significant ✅
I vs S_C: p = 0.00226 → Significant ✅
I vs U: p = 0.18772 → Not significant ❌
M vs N: p = 0.00217 → Significant ✅
M vs O: p = 0.02782 → Significant ✅
M vs P: p = 0.00012 → Significant ✅
M vs R: p = 0.02222 → Significant ✅
M vs S_C: p = 0.12413 → Not significant ❌
M vs U: p = 0.15921 → Not significant ❌
N v

In [5]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

# Read filtered contingency table from Excel file
contingency_filtered = pd.read_excel(r'D:\melikeh\tweets\contingency_filtered.xlsx', index_col=0)

# Calculate Cramér's V
chi2, p, dof, expected = chi2_contingency(contingency_filtered)

n = contingency_filtered.values.sum()
r, k = contingency_filtered.shape

cramers_v = np.sqrt(chi2 / (n * min(r-1, k-1)))

print(f"Cramér's V: {cramers_v:.3f}")
print(f"Chi-square: {chi2:.2f}, p = {p:.6f}")

Cramér's V: 0.442
Chi-square: 225.95, p = 0.000000
